# Multi-Document Research Contradiction Detector

Parse **5 landmark NLP papers** with LlamaParse, index them all into a single Moss in-process store,
then run a `FunctionAgent` that detects contradictions across papers with sub-10ms retrieval.

```mermaid
flowchart TD
    A[5 arXiv PDFs]
    B[LlamaParse<br/>(parse_page_with_agent)]
    C[documents]
    D[SentenceSplitter]
    E[nodes<br/>(paper metadata injected)]
    F[MossToolSpec<br/>(single index, all 5 papers)]
    G[list_papers<br/>(FunctionTool)]
    H[FunctionAgent]
    I[Contradiction Q&A<br/>with citations]

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    E --> G
    F --> H
    G --> H
    H --> I
```

### Known contradictions in the corpus

| Topic | Paper A claim | Paper B contradiction |
|-------|--------------|----------------------|
| NSP training objective | BERT: NSP improves downstream tasks | RoBERTa: NSP hurts — remove it |
| NSP replacement | RoBERTa: no sentence-pair task needed | ALBERT: SOP is better than NSP |
| Masking strategy | BERT: static masking is sufficient | RoBERTa: dynamic masking outperforms static |
| Pretraining objective | BERT: masked LM is effective | XLNet: independence assumption causes pretrain/finetune mismatch |
| Model size vs efficiency | General: more params = better | ALBERT: parameter sharing + factorized embeddings beat naïve scaling |

## 1 · Install

In [1]:
%pip install -q \
    llama-cloud-services \
    llama-index-tools-moss \
    llama-index-core \
    llama-index-llms-openai \
    inferedge-moss \
    openai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2 · LLM Settings

Contradiction detection requires strong multi-step reasoning.
`gpt-4.1` gives best results; `gpt-4.1-mini` works but may miss subtle conflicts.
No embed model needed — Moss handles embeddings internally.

In [2]:
import os
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4.1", api_key=os.getenv("OPENAI_API_KEY"))
print("LLM:", Settings.llm.model)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: gpt-4.1


## 3 · Paper Corpus

Five papers chosen for documented, research-community-acknowledged contradictions.
All are publicly available on arXiv — no login required.

In [3]:
PAPERS = [
    {
        "id":            "bert",
        "title":         "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors_short": "Devlin et al.",
        "year":          2018,
        "arxiv_id":      "1810.04805",
    },
    {
        "id":            "roberta",
        "title":         "RoBERTa: A Robustly Optimized BERT Pretraining Approach",
        "authors_short": "Liu et al.",
        "year":          2019,
        "arxiv_id":      "1907.11692",
    },
    {
        "id":            "xlnet",
        "title":         "XLNet: Generalized Autoregressive Pretraining for Language Understanding",
        "authors_short": "Yang et al.",
        "year":          2019,
        "arxiv_id":      "1906.08237",
    },
    {
        "id":            "albert",
        "title":         "ALBERT: A Lite BERT for Self-supervised Learning of Language Representations",
        "authors_short": "Lan et al.",
        "year":          2019,
        "arxiv_id":      "1909.11942",
    },
    {
        "id":            "attention",
        "title":         "Attention Is All You Need",
        "authors_short": "Vaswani et al.",
        "year":          2017,
        "arxiv_id":      "1706.03762",
    },
]

print(f"{len(PAPERS)} papers in corpus:")
for p in PAPERS:
    print(f"  [{p['authors_short']}, {p['year']}] {p['title']}")

5 papers in corpus:
  [Devlin et al., 2018] BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
  [Liu et al., 2019] RoBERTa: A Robustly Optimized BERT Pretraining Approach
  [Yang et al., 2019] XLNet: Generalized Autoregressive Pretraining for Language Understanding
  [Lan et al., 2019] ALBERT: A Lite BERT for Self-supervised Learning of Language Representations
  [Vaswani et al., 2017] Attention Is All You Need


## 4 · Download Papers from arXiv

In [4]:
import urllib.request

os.makedirs("papers", exist_ok=True)

for paper in PAPERS:
    dest = f"papers/{paper['id']}.pdf"
    if os.path.exists(dest):
        print(f"  ✓ {paper['id']}.pdf already present")
        continue
    url = f"https://arxiv.org/pdf/{paper['arxiv_id']}.pdf"
    print(f"  ⬇  Downloading {paper['id']} from arXiv…", end=" ", flush=True)
    urllib.request.urlretrieve(url, dest)
    size_kb = os.path.getsize(dest) // 1024
    print(f"{size_kb} KB")

print("\nAll papers ready.")

  ✓ bert.pdf already present
  ✓ roberta.pdf already present
  ✓ xlnet.pdf already present
  ✓ albert.pdf already present
  ✓ attention.pdf already present

All papers ready.


## 5 · Parse Papers with LlamaParse

Each PDF is parsed once and cached to `paper_cache/` as markdown.
Subsequent runs load from cache — skips ~60 s parse time per paper.

Same LlamaParse settings as `llama_moss_agent.ipynb`:
- `parse_page_with_agent` — agent-driven layout understanding
- `high_res_ocr` — accuracy on complex multi-column layouts
- `output_tables_as_HTML` — richer table representation

In [11]:
from llama_cloud_services import LlamaParse

parser = LlamaParse(
    parse_mode="parse_page_with_agent",
    model="openai-gpt-4-1-mini",
    high_res_ocr=True,
    adaptive_long_table=True,
    outlined_table_extraction=True,
    output_tables_as_HTML=True,
)

print("LlamaParse ready")

LlamaParse ready


In [ ]:
import time
from pathlib import Path
from llama_index.core import Document

CACHE_DIR = Path("paper_cache")
CACHE_DIR.mkdir(exist_ok=True)

all_documents: dict[str, list[Document]] = {}

for paper in PAPERS:
    cache_path = CACHE_DIR / f"{paper['id']}.md"

    if cache_path.exists():
        text = cache_path.read_text()
        all_documents[paper["id"]] = [Document(text=text)]
        print(f"  {paper['id']:<12} loaded from cache  ({len(text):>8,} chars)")
    else:
        print(f"  {paper['id']:<12} parsing with LlamaParse…")
        t0 = time.perf_counter()
        result = await parser.aparse(f"papers/{paper['id']}.pdf")
        docs = result.get_markdown_documents(split_by_page=False)
        all_documents[paper["id"]] = docs
        combined = "\n\n".join(d.text for d in docs)
        cache_path.write_text(combined)
        elapsed = time.perf_counter() - t0
        print(f"   {paper['id']:<12} parsed in {elapsed:.1f}s  ({len(combined):>8,} chars)")

total = sum(sum(len(d.text) for d in docs) for docs in all_documents.values())
print(f"\nCorpus ready: {total:,} total chars across {len(PAPERS)} papers")

  ✓ bert         loaded from cache  (  72,761 chars)
  ✓ roberta      loaded from cache  (  56,642 chars)
  ✓ xlnet        loaded from cache  (  72,253 chars)
  ✓ albert       loaded from cache  (  72,111 chars)
  ✓ attention    loaded from cache  (  50,479 chars)

Corpus ready: 324,246 total chars across 5 papers


In [7]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1536, chunk_overlap=200)
all_nodes = []

for paper in PAPERS:
    nodes = splitter.get_nodes_from_documents(all_documents[paper["id"]])

    # Inject paper identity into every node so Moss results are always attributed
    for node in nodes:
        node.metadata.update({
            "paper_id":      paper["id"],
            "title":         paper["title"],
            "authors_short": paper["authors_short"],
            "year":          str(paper["year"]),
            "arxiv_id":      paper["arxiv_id"],
        })

    all_nodes.extend(nodes)
    print(f"  {paper['authors_short']}, {paper['year']:>4d}: {len(nodes):>3d} nodes")

print(f"\nTotal nodes: {len(all_nodes)}")

  Devlin et al., 2018:  16 nodes
  Liu et al., 2019:  12 nodes
  Yang et al., 2019:  18 nodes
  Lan et al., 2019:  19 nodes
  Vaswani et al., 2017:  11 nodes

Total nodes: 76


## 6 · Moss — Single Index Across the Whole Corpus

All papers go into **one Moss index**. Each chunk's text is prefixed with a citation header
so the LLM always knows the source even if Moss strips metadata from results.

```
DocumentInfo.text =
  "SOURCE: Devlin et al., 2018 — BERT: Pre-training...\n\n<chunk text>"
```

**Query options**

| Setting | Values |
|---------| ---------------|
| `top_k` |  **6** — more cross-paper context |
| `alpha` | **0.6** — more BM25 weight to surface exact terms like "NSP", "SOP", "dynamic masking" |

In [12]:
from inferedge_moss import MossClient, DocumentInfo
from llama_index.tools.moss import MossToolSpec, QueryOptions

client = MossClient(
    project_id=os.environ["MOSS_PROJECT_ID"],
    project_key=os.environ["MOSS_PROJECT_KEY"],
)

moss_tool = MossToolSpec(
    client=client,
    index_name="research-corpus",
    query_options=QueryOptions(
        top_k=6,    # more chunks → better cross-paper coverage per query
        alpha=0.6,  # 60% semantic + 40% BM25 (exact technical term recall)
    ),
)

print("MossToolSpec ready  •  index=research-corpus  •  top_k=6  •  alpha=0.6")

MossToolSpec ready  •  index=research-corpus  •  top_k=6  •  alpha=0.6


In [9]:
# Prefix every chunk with its citation so the LLM always sees the source
docs_for_moss = [
    DocumentInfo(
        id=node.node_id,
        text=(
            f"SOURCE: {node.metadata['authors_short']}, {node.metadata['year']} "
            f"\u2014 {node.metadata['title']}\n\n"
            f"{node.get_content()}"
        ),
        metadata={k: str(v) for k, v in node.metadata.items()},
    )
    for node in all_nodes
]

print(f"Prepared {len(docs_for_moss)} DocumentInfo objects")
print(f"\nSample header (first 200 chars):")
print(docs_for_moss[0].text[:200])

Prepared 76 DocumentInfo objects

Sample header (first 200 chars):
SOURCE: Devlin et al., 2018 — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

arXiv:1810.04805v2 [cs.CL] 24 May 2019

# BERT: Pre-training of Deep Bidirectional Trans


In [19]:
print(f"Indexing {len(docs_for_moss)} chunks into Moss…")
t0 = time.perf_counter()

# await moss_tool.index_docs(docs_for_moss)

print(f"Indexed in {time.perf_counter()-t0:.2f}s")

Indexing 76 chunks into Moss…
Indexed in 0.00s


In [13]:
print("Loading into Moss in-process store…")
t0 = time.perf_counter()

await moss_tool._load_index()

print(f"\u2705 Loaded in {(time.perf_counter()-t0)*1000:.0f} ms  (one-time cost)")
print("   All subsequent agent queries run in-process at 1\u201310 ms")

moss_tools = moss_tool.to_tool_list()
print(f"\nMoss tools exposed to agent: {[t.metadata.name for t in moss_tools]}")

Loading into Moss in-process store…
✅ Loaded in 4358 ms  (one-time cost)
   All subsequent agent queries run in-process at 1–10 ms

Moss tools exposed to agent: ['query']


## 7 · list_papers Tool

A lightweight `FunctionTool` that returns the corpus inventory.
Gives the agent a map of what's available before it starts querying Moss.

In [14]:
from llama_index.core.tools import FunctionTool

def list_available_papers() -> str:
    """List all research papers indexed in the corpus with their metadata."""
    lines = [f"Research corpus ({len(PAPERS)} papers):\n"]
    for p in PAPERS:
        lines.append(f"  [{p['authors_short']}, {p['year']}] {p['title']}")
        lines.append(f"    arXiv: {p['arxiv_id']}\n")
    return "\n".join(lines)

list_papers_tool = FunctionTool.from_defaults(
    fn=list_available_papers,
    name="list_papers",
    description=(
        "Returns metadata for all papers in the research corpus. "
        "Call this first to know which papers are available before issuing search queries."
    ),
)

print(list_available_papers())

Research corpus (5 papers):

  [Devlin et al., 2018] BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
    arXiv: 1810.04805

  [Liu et al., 2019] RoBERTa: A Robustly Optimized BERT Pretraining Approach
    arXiv: 1907.11692

  [Yang et al., 2019] XLNet: Generalized Autoregressive Pretraining for Language Understanding
    arXiv: 1906.08237

  [Lan et al., 2019] ALBERT: A Lite BERT for Self-supervised Learning of Language Representations
    arXiv: 1909.11942

  [Vaswani et al., 2017] Attention Is All You Need
    arXiv: 1706.03762



## 8 · Contradiction Detector Agent

The agent gets two tools:

| Tool | Source | Purpose |
|------|--------|---------|
| `list_papers` | `FunctionTool` | Orient the agent to the available corpus |
| `query` | `MossToolSpec` | Hybrid semantic + BM25 retrieval across all 5 papers |

The system prompt instructs the agent to:
- Always attribute claims with inline `[Author et al., Year]` citations
- Issue multiple Moss queries per question (claim + negation + related terms)
- Structure contradictions as A claims X vs B argues Y
- Classify each conflict as direct contradiction / methodological difference / scope difference

In [15]:
from llama_index.core.agent import FunctionAgent

paper_list_str = "\n".join(
    f"  - [{p['authors_short']}, {p['year']}] {p['title']}" for p in PAPERS
)

SYSTEM_PROMPT = f"""You are a research analyst specialising in identifying contradictions,
disagreements, and tensions across academic papers.

## Corpus ({len(PAPERS)} papers)
{paper_list_str}

## Rules
1. **Always cite inline** — every factual claim must include [AuthorLastName et al., Year].
2. **Search exhaustively** — issue multiple Moss queries per topic:
   the claim, its negation, and related terms.
   Example for NSP: query "next sentence prediction" THEN "remove NSP" THEN "sentence order prediction".
3. **Structure each contradiction** as:
   > **[Paper A, Year]** claims: *exact quote or close paraphrase*
   > **[Paper B, Year]** contradicts: *exact quote or close paraphrase* — because [reason/evidence]
4. **Classify each conflict** as one of:
   - Direct contradiction (same experimental setting, opposite conclusions)
   - Methodological difference (different configs/data explain different results)
   - Scope difference (both correct, but in different contexts)
5. Never assert a contradiction without citing evidence from BOTH sides found in search results.
6. Call list_papers first if unsure what papers are in the corpus.
"""

tools = [
    list_papers_tool,
    *moss_tool.to_tool_list(),
]

agent = FunctionAgent(
    tools=tools,
    llm=Settings.llm,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Contradiction Detector Agent ready with {len(tools)} tools:")
for t in tools:
    print(f"  \u2022 {t.metadata.name}")

Contradiction Detector Agent ready with 2 tools:
  • list_papers
  • query


In [16]:
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import ToolCall, ToolCallResult

ctx = Context(agent)   # maintains conversation state across turns

async def chat(question: str) -> None:
    print(f"Q: {question}")

    handler = agent.run(question, ctx=ctx)

    async for ev in handler.stream_events():
        if isinstance(ev, ToolCall):
            print(f"  [tool call]  {ev.tool_name}({ev.tool_kwargs})")
        elif isinstance(ev, ToolCallResult):
            preview = str(ev.tool_output)[:300].replace("\n", " ")
            print(f"  [tool result] {ev.tool_name} \u2192 {preview}\u2026")

    resp = await handler
    print(f"\nA: {resp}")

## 9 · Demo Queries

Each query targets a known contradiction in the corpus.
Watch the `[tool call]` lines — the agent should issue **multiple** Moss queries per question
to gather evidence from both sides before concluding.

Because all 5 papers are in a single Moss in-process store, each retrieval is <10 ms —
the agent can do 6–8 hops and still return faster than a single cloud vector DB call.

In [17]:
# NSP: BERT says it helps → RoBERTa says it hurts → ALBERT replaces it with SOP
await chat(
    "What do these papers say about the Next Sentence Prediction (NSP) training objective? "
    "Are there direct contradictions between the papers?"
)

Q: What do these papers say about the Next Sentence Prediction (NSP) training objective? Are there direct contradictions between the papers?
  [tool call]  query({'query': 'next sentence prediction'})
  [tool call]  query({'query': 'remove NSP'})
  [tool call]  query({'query': 'sentence order prediction'})
  [tool result] query → Search results for: 'next sentence prediction'  Match 1 [Score: 0.60] Source: Unknown Source (Page: N/A) Content: SOURCE: Lan et al., 2019 — ALBERT: A Lite BERT for Self-supervised Learning of Language Representations  ---  Published as a conference paper at ICLR 2020  transformer. Very recently, Ba…
  [tool result] query → Search results for: 'remove NSP'  Match 1 [Score: 1.00] Source: Unknown Source (Page: N/A) Content: SOURCE: Liu et al., 2019 — RoBERTa: A Robustly Optimized BERT Pretraining Approach  The NSP loss was hypothesized to be an important factor in training the original BERT model. Devlin et al. (2019) ob…
  [tool result] query → Search results f

In [20]:
# Masking: BERT static masking vs RoBERTa dynamic masking
await chat(
    "Do the papers agree on the best masking strategy during pre-training? "
    "Identify any contradictions and classify them."
)

Q: Do the papers agree on the best masking strategy during pre-training? Identify any contradictions and classify them.
  [tool call]  query({'query': 'masking strategy'})
  [tool call]  query({'query': 'dynamic masking'})
  [tool call]  query({'query': 'static masking'})
  [tool call]  query({'query': 'masking during pretraining'})
  [tool result] query → Search results for: 'masking strategy'  Match 1 [Score: 1.00] Source: Unknown Source (Page: N/A) Content: SOURCE: Devlin et al., 2018 — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding  This allows us to answer the following questions:  1. **Question:** Does BERT real…
  [tool result] query → Search results for: 'dynamic masking'  Match 1 [Score: 0.88] Source: Unknown Source (Page: N/A) Content: SOURCE: Devlin et al., 2018 — BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding  This allows us to answer the following questions:  1. **Question:** Does BERT reall…
  [tool resul

In [ ]:
# Pretraining objectives: which papers challenge BERT's masked LM approach and why?
await chat(
    "Compare the pre-training objectives across all papers. "
    "Which papers directly challenge BERT's masked language model approach, and on what grounds?"
)

In [ ]:
# Follow-up — multi-hop, ctx carries conversation state
await chat(
    "Of the contradictions you found, which is best supported by ablation experiments "
    "rather than just end-task benchmark comparisons?"
)